In [1]:
import pandas as pd


In [2]:
data = pd.read_csv("IMDB Dataset.csv")
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [3]:
data.isnull().sum()

review       0
sentiment    0
dtype: int64

In [4]:
data.drop_duplicates(inplace = True)

In [5]:
data.shape

(49582, 2)

### Test Preprocessing

In [6]:
# converting to lowercase
data["review"] = data["review"].str.lower()

In [7]:
# removing the URL - import regex - built in library for regular expression
import re


In [8]:
def remove_urls(text):
    text = re.sub(r"http\S+" , "" , text)
    return text

data["review"] = data["review"].apply(remove_urls)

In [9]:
def remove_tages(text):
    text = re.sub(r"<.*?>" , "", text) # A-Z a-z 0-9 \s
    return text

In [10]:
def remove_punctuations(text):
    text = re.sub(r"[^A-Za-z0-9\s]" , "", text) # A-Z a-z 0-9 \s
    return text

data["review"] = data["review"].apply(remove_punctuations)

In [11]:
data.head()

,review,sentiment
0,one of the other reviewers has mentioned that ...,positive
1,a wonderful little production br br the filmin...,positive
2,i thought this was a wonderful way to spend ti...,positive
3,basically theres a family where a little boy j...,negative
4,petter matteis love in the time of money is a ...,positive


In [66]:
# remove stop words
%pip install nltk


   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------ --------------------------------- 0.3/1.6 MB ? eta -:--:--
   ------------- -------------------------- 0.5/1.6 MB 698.6 kB/s eta 0:00:02
   -------------------- ------------------- 0.8/1.6 MB 854.1 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 844.9 kB/s eta 0:00:01
   --------------------------- ------------ 1.0/1.6 MB 844.9 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 822.0 kB/s eta 0:00:01
   --------------------------------- ------ 1.3/1.6 MB 822.0 kB/s eta 0:00:01
   ---------------------------------------- 1.6/1.6 MB 811.9 kB/s  0:00:02

   ---------------------------------------- 0/4 [tqdm]
   ---------------------------------------- 0/4 [tqdm]
 

In [12]:
import nltk

In [85]:
nltk.download("punkt")
nltk.download("punkt_tab")
nltk.download("stopwords")

[nltk_data] Downloading package punkt to C:\Users\Syed wasif
[nltk_data]     Hussain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to C:\Users\Syed wasif
[nltk_data]     Hussain\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to C:\Users\Syed wasif
[nltk_data]     Hussain\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [13]:
# Removing the stop words
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords

def remove_stopowords(text):
    tokens = word_tokenize(text)
    stop_words = stopwords.words("english")
    for word in tokens:
        if word in stop_words:
            text = text.replace(word, "")
    return text

data["review"] = data["review"].apply(remove_stopowords)

In [14]:
# we will perform stemming

from nltk.stem import PorterStemmer

def stemming(text):
    ps = PorterStemmer()
    stemmed_words = []

    tokens = word_tokenize(text)
    for token in tokens:
        stemmed_token = ps.stem(token)
        stemmed_words.append(stemmed_token)

    return " ".join(stemmed_words)

data["review"] = data["review"].apply(stemming)

data.head()

,review,sentiment
0,e revew nted wtchg 1 oz epod ll hook y rght ex...,positive
1,wder ltle producti br br film techniqu unssum ...,positive
2,thought th wder wy spend tme o hot summer week...,positive
3,bsclli re fmli lttle boy jke thk re zomb close...,negative
4,petter mtte love time mey vulli stunng film wt...,positive


 ### Encoding

In [15]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
data["sentiment"] = le.fit_transform(data["sentiment"])

In [16]:
y = data["sentiment"]
y

0        1
1        1
2        1
3        0
4        1
        ..
49995    1
49996    0
49997    0
49998    0
49999    0
Name: sentiment, Length: 49582, dtype: int64

### vecorisation

In [17]:
from sklearn.feature_extraction.text import TfidfVectorizer

tf = TfidfVectorizer(max_features=5000)

X = tf.fit_transform(data["review"])

### Dataset and DataLoader

In [18]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [19]:
X_test.shape

(9917, 5000)

In [20]:
# now we will convert the sparse matrix into araaya

X_train = X_train.toarray()
X_test = X_test.toarray()

In [22]:
# now convert the array to tensor dataset
import torch
from torch.utils.data import TensorDataset, DataLoader

train_set = TensorDataset(
    torch.from_numpy(X_train).float(),
    torch.from_numpy(y_train.values).float()
)

test_set = TensorDataset(
    torch.from_numpy(X_test).float(),
    torch.from_numpy(y_test.values).float()
)


In [23]:
train_loader = DataLoader(train_set, shuffle=True, batch_size=64)
test_loader = DataLoader(test_set, shuffle=True, batch_size=64)

### Building our own RNN

In [24]:
import torch.nn as nn
import torch.optim as optim

In [26]:
class RNN(nn.Module):
    def __init__(self , input_size, hidden_size=128, num_layers=1):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers

        self.rnn = nn.RNN(input_size, hidden_size, num_layers, batch_first=True)

        # fully connected layer
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # optional => shape (num of layers, batch size, hidden size) - if not given then also it willl start fgrtom 0
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size)

        out, _ = self.rnn(x, h0) 
        # 1st value = hidden state of all the timesteps => (batch, seq_len, hidden size)
        # 2nd value = final hidden state of last timestep

        out = self.fc(out[:, -1, :])
        return out

In [29]:
input_size = X_train.shape[1]
model = RNN(input_size)
criterion = nn.BCELoss()  # loss function - binary cross entopy loss
optimizer = optim.Adam(model.parameters())

### Traning the RNN

In [30]:
epochs = 10
for epoch in range(epochs):
    model.train()

    for Xb, yb in train_loader:
        optimizer.zero_grad()

        Xb = Xb.unsqueeze(1) # add singleton direction
        
        outputs = model(Xb) # (batch_size, 1)

        outputs = torch.sigmoid(outputs.squeeze()) # (batch_size,) => probability

        loss = criterion(outputs, yb) # compute loss
        loss.backward() # backprop
        optimizer.step() # weights update

    print(f"epoch = {epoch+1}/{epochs} and loss = {loss.item()}")
    

epoch = 1/10 and loss = 0.22622166574001312
epoch = 2/10 and loss = 0.35604703426361084
epoch = 3/10 and loss = 0.3485235571861267
epoch = 4/10 and loss = 0.3192480504512787
epoch = 5/10 and loss = 0.34450000524520874
epoch = 6/10 and loss = 0.20703689754009247
epoch = 7/10 and loss = 0.19222798943519592
epoch = 8/10 and loss = 0.3225046694278717
epoch = 9/10 and loss = 0.468772292137146
epoch = 10/10 and loss = 0.29082316160202026


# evaluate

model.eval()

with torch.no_grad():
    correct_vals = 0
    tot_vals = 0
    
    for Xb, yb in test_loader:
        Xb = Xb.unsqueeze(1)

        outputs = model(Xb)
        predicted = (torch.sigmoid(outputs.squeeze()) > 0.5).float()

        tot_vals += yb.size(0)
        correct_vals += (predicted == yb).sum().item()

    print(f"accuracy = {correct_vals/tot_vals*100}")

### Checking the output with the new data

In [43]:

user_review = "This movie http://www.movie.com was very bad! The acting was poor and the plot kept me sleeping."

# text - preprocessing

user_review = user_review.lower()
Cleaned_review = remove_urls(user_review)
Cleaned_review = remove_punctuations(Cleaned_review)
Cleaned_review = remove_stopowords(Cleaned_review)
Cleaned_review = stemming(Cleaned_review)

print(f"Cleaned review : {Cleaned_review}")

Cleaned review : movi bad act poor plot kept sleep


In [44]:
X_tfidf = tf.transform([Cleaned_review]) # this covert the string into sparse matrix
X_input = torch.tensor(X_tfidf.toarray() , dtype=torch.float32)
X_input = X_input.unsqueeze(1)

In [45]:
model.eval()
with torch.no_grad():
    output = model(X_input)
    probability = torch.sigmoid(output.squeeze()).item()

print(f"Probability Score: {probability:.4f}")
print("Prediction:", "POSITIVE" if probability > 0.5 else "NEGATIVE")

Probability Score: 0.0005
Prediction: NEGATIVE
